In [17]:
import sys
from pathlib import Path

print("EXECUTABLE:", sys.executable)
print("CWD:", Path.cwd())
print("PATH0:", sys.path[0])

EXECUTABLE: C:\Users\Gabriela\Pmonkey\IA\predicao\.venv\Scripts\python.exe
CWD: c:\Users\Gabriela\Pmonkey\IA\predicao\notebooks
PATH0: C:\Users\Gabriela\AppData\Local\Programs\Python\Python312\python312.zip


In [18]:
import pandas as pd
from pathlib import Path

# Obter paths do projeto
from src.config import ROOT_DIR, RAW_DIR, INTERIM_DIR

# Ajuste para um dos seus arquivos
arquivo = RAW_DIR / "202201_Licitacoes" / "202201_Licitacao.csv"

# Os dados do governo geralmente vêm em latin-1 com ;
df_test = pd.read_csv(arquivo, sep=";", encoding="latin-1", nrows=5)
print(df_test.columns.tolist())
df_test.head()

# Exibe o diretório de trabalho para depuração
print("CWD:", Path.cwd())
print("ROOT_DIR:", ROOT_DIR)
print("INTERIM_DIR:", INTERIM_DIR)

# Leitura das estruturas de dados (schemas)
df_licitacao = pd.read_parquet(INTERIM_DIR / "licitacao.parquet")
df_participantes = pd.read_parquet(INTERIM_DIR / "participanteslicitacao.parquet")

# Exibição dos rótulos de colunas para identificação de chaves estruturais
print("Campos - Licitação:", df_licitacao.columns.tolist())
print("Campos - Participantes:", df_participantes.columns.tolist())

# Carregamento dos dados analíticos (redundante em células seguintes, mas útil aqui)
df_licitacao = pd.read_parquet(INTERIM_DIR / "licitacao.parquet")

# Definição da chave de entidade
chave_composta = ['Número Licitação', 'Código UG']

# Avaliação de duplicidade estrutural
total_registros = len(df_licitacao)
total_duplicados = df_licitacao.duplicated(subset=chave_composta).sum()

print(f"Volume total de registros analisados: {total_registros:,}")
print(f"Registros violando a restrição de unicidade: {total_duplicados:,}")

if total_duplicados > 0:
    print("\nExemplo de registros duplicados detectados (amostra):")
    df_duplicados = df_licitacao[df_licitacao.duplicated(subset=chave_composta, keep=False)]
    print(df_duplicados[chave_composta + ['ano_mes_arquivo']].sort_values(by=chave_composta).head(4))

['Número Licitação', 'Código UG', 'Nome UG', 'Código Modalidade Compra', 'Modalidade Compra', 'Número Processo', 'Objeto', 'Situação Licitação', 'Código Órgão Superior', 'Nome Órgão Superior', 'Código Órgão', 'Nome Órgão', 'UF', 'Município', 'Data Resultado Compra', 'Data Abertura', 'Valor Licitação']
CWD: c:\Users\Gabriela\Pmonkey\IA\predicao\notebooks
ROOT_DIR: C:\Users\Gabriela\Pmonkey\IA\predicao
INTERIM_DIR: C:\Users\Gabriela\Pmonkey\IA\predicao\data\interim
Campos - Licitação: ['Número Licitação', 'Código UG', 'Nome UG', 'Código Modalidade Compra', 'Modalidade Compra', 'Número Processo', 'Objeto', 'Situação Licitação', 'Código Órgão Superior', 'Nome Órgão Superior', 'Código Órgão', 'Nome Órgão', 'UF', 'Município', 'Data Resultado Compra', 'Data Abertura', 'Valor Licitação', 'ano_mes_arquivo']
Campos - Participantes: ['Número Licitação', 'Código UG', 'Nome UG', 'Código Modalidade Compra', 'Modalidade Compra', 'Número Processo', 'Código Órgão', 'Nome Órgão', 'Código Item Compra', '

In [19]:
# Pegar uma licitação que aparece em múltiplos meses
exemplo = df_licitacao[
    (df_licitacao["Número Licitação"] == "000012021") &
    (df_licitacao["Código UG"] == "160220")
]

print(f"Aparições da mesma chave: {len(exemplo)}")
print()
exemplo.T  # transposto fica mais legível com muitas colunas

Aparições da mesma chave: 2



,6,25938
Número Licitação,000012021,000012021
Código UG,160220,160220
Nome UG,COMISSAO REGIONAL DE OBRAS/5,COMISSAO REGIONAL DE OBRAS/5
Código Modalidade Compra,9999,9997
Modalidade Compra,Pregão - Registro de Preço,Concorrência - Registro de Preço
Número Processo,64328003794202196,64328001396202054
Objeto,Objeto: Pregão Eletrônico - Aquisição de Mater...,Objeto: Contratação de Estudos e Projetos de E...
Situação Licitação,Publicado,Evento de Resultado de Julgame
Código Órgão Superior,52000,52000
Nome Órgão Superior,Ministério da Defesa,Ministério da Defesa


In [20]:
chave_v2 = ['Número Licitação', 'Código UG', 'Código Modalidade Compra']

total = len(df_licitacao)
duplicados_v2 = df_licitacao.duplicated(subset=chave_v2).sum()

print(f"Total de registros: {total:,}")
print(f"Duplicatas com chave_v2: {duplicados_v2:,}")
print(f"% duplicatas: {duplicados_v2/total*100:.2f}%")

Total de registros: 109,346
Duplicatas com chave_v2: 0
% duplicatas: 0.00%


In [21]:

# Pegar um caso onde a chave_v2 ainda dá duplicata
duplicados_idx = df_licitacao[df_licitacao.duplicated(subset=chave_v2, keep=False)].index
if len(duplicados_idx) > 0:
    primeiro = df_licitacao.loc[duplicados_idx].iloc[0]
    grupo = df_licitacao[
        (df_licitacao['Número Licitação'] == primeiro['Número Licitação']) &
        (df_licitacao['Código UG'] == primeiro['Código UG']) &
        (df_licitacao['Código Modalidade Compra'] == primeiro['Código Modalidade Compra'])
    ]
    print(grupo[['Número Processo', 'Objeto', 'Valor Licitação', 
                 'Situação Licitação', 'ano_mes_arquivo']].T)

In [22]:
df_itens = pd.read_parquet(INTERIM_DIR / "itemlicitacao.parquet")
df_empenhos = pd.read_parquet(INTERIM_DIR / "empenhosrelacionados.parquet")

print("Campos - Itens:", df_itens.columns.tolist())
print(f"Linhas - Itens: {len(df_itens):,}\n")

print("Campos - Empenhos:", df_empenhos.columns.tolist())
print(f"Linhas - Empenhos: {len(df_empenhos):,}")

Campos - Itens: ['Número Licitação', 'Código UG', 'Nome UG', 'Código Modalidade Compra', 'Modalidade Compra', 'Número Processo', 'Código Órgão', 'Nome Órgão', 'Código Item Compra', 'Descrição', 'Quantidade Item', 'Valor Item', 'Código Vencedor', 'Nome Vencedor', 'ano_mes_arquivo']
Linhas - Itens: 949,377

Campos - Empenhos: ['Número Licitação', 'Código UG', 'Nome UG', 'Código Modalidade Compra', 'Modalidade Compra', 'Número Processo', 'Código Empenho', 'Data Emissão Empenho', 'Observação Empenho', 'Valor Empenho (R$)', 'ano_mes_arquivo']
Linhas - Empenhos: 1,478


In [23]:
total = len(df_licitacao)
nulls_processo = df_licitacao['Número Processo'].isnull().sum()
duplicados_processo = df_licitacao.duplicated(subset=['Número Processo']).sum()

print(f"Total: {total:,}")
print(f"Nulls em Número Processo: {nulls_processo:,}")
print(f"Duplicatas em Número Processo: {duplicados_processo:,}")

Total: 109,346
Nulls em Número Processo: 0
Duplicatas em Número Processo: 10,520


In [24]:
# 1. Como estão exatamente os valores monetários?
print("Amostras de Valor Licitação:")
print(df_licitacao["Valor Licitação"].dropna().head(10).tolist())

# 2. Datas estão todas no formato DD/MM/AAAA?
print("\nAmostras de Data Abertura:")
print(df_licitacao["Data Abertura"].dropna().head(10).tolist())

# 3. Quantos nulls existem nas colunas críticas?
print("\nNulls por coluna em Licitacao:")
print(df_licitacao.isnull().sum().sort_values(ascending=False))

Amostras de Valor Licitação:
['703363,4400', '1520,0000', '249,9100', '945213,9000', '10523,0000', '666213,8100', '63661,5000', '9886,2000', '510559,5600', '44000,0000']

Amostras de Data Abertura:
['29/12/2021', '13/01/2022', '20/01/2022', '29/11/2021', '13/12/2021', '20/09/2021', '14/12/2021', '17/12/2021', '26/01/2022', '28/12/2021']

Nulls por coluna em Licitacao:
Data Abertura               72362
Número Licitação                0
Código UG                       0
Nome UG                         0
Modalidade Compra               0
Código Modalidade Compra        0
Objeto                          0
Situação Licitação              0
Código Órgão Superior           0
Número Processo                 0
Nome Órgão Superior             0
Código Órgão                    0
UF                              0
Nome Órgão                      0
Município                       0
Data Resultado Compra           0
Valor Licitação                 0
ano_mes_arquivo                 0
dtype: int64


In [25]:
# 1. Nulos de Data Abertura por modalidade
nulls_por_modalidade = (
    df_licitacao
    .groupby("Modalidade Compra")["Data Abertura"]
    .agg(total="count", nulls=lambda s: s.isnull().sum())
)
nulls_por_modalidade["pct_nulls"] = (
    nulls_por_modalidade["nulls"] / 
    (nulls_por_modalidade["total"] + nulls_por_modalidade["nulls"]) * 100
).round(1)
print("Nulls de Data Abertura por modalidade:")
print(nulls_por_modalidade.sort_values("pct_nulls", ascending=False))

# 2. Nulos de Data Abertura por mês
print("\nNulls de Data Abertura por mês:")
print(
    df_licitacao
    .groupby("ano_mes_arquivo")["Data Abertura"]
    .apply(lambda s: s.isnull().sum())
)

Nulls de Data Abertura por modalidade:
                                  total  nulls  pct_nulls
Modalidade Compra                                        
Concorrência                          0    132      100.0
Concorrência - Registro de Preço      0      6      100.0
Concorrência Internacional            0      2      100.0
Concurso                              0     12      100.0
Convite                               0     27      100.0
Inexigibilidade de Licitação          0  18327      100.0
Tomada de Preços                      0    501      100.0
Dispensa de Licitação              6310  53355       89.4
Pregão                            12448      0        0.0
Pregão - Registro de Preço        18226      0        0.0

Nulls de Data Abertura por mês:
ano_mes_arquivo
202201    1984
202202    3806
202203    5752
202204    5173
202205    7658
202206    7195
202207    6507
202208    7486
202209    6786
202210    6299
202211    7432
202212    6284
Name: Data Abertura, dtype: int64


In [26]:
print(df_participantes["Flag Vencedor"].value_counts(dropna=False))


Flag Vencedor
NÃO    3489950
SIM     949377
Name: count, dtype: int64


In [27]:
import pandas as pd

from src.config import INTERIM_DIR

df_lic = pd.read_parquet(INTERIM_DIR / "licitacao_tratado.parquet")
df_par = pd.read_parquet(INTERIM_DIR / "participanteslicitacao_tratado.parquet")

# 1. Tipos: valor virou float? datas viraram datetime?
print("Tipos de Licitacao:")
print(df_lic.dtypes)
print()

# 2. Range de valores faz sentido?
print(f"Valor Licitação: min={df_lic['valor_licitacao'].min():,.2f}, "
      f"max={df_lic['valor_licitacao'].max():,.2f}, "
      f"mediana={df_lic['valor_licitacao'].median():,.2f}")

# 3. Flag vencedor convertido corretamente?
print("\nFlag Vencedor após tratamento:")
print(df_par["flag_vencedor"].value_counts(dropna=False))
# Esperado: True 949377, False 3489950

# 4. Chave única confirmada?
chave = ["numero_licitacao", "codigo_ug", "codigo_modalidade_compra"]
print(f"\nDuplicatas na chave: {df_lic.duplicated(subset=chave).sum()}")
# Esperado: 0

Tipos de Licitacao:
numero_licitacao                       str
codigo_ug                              str
nome_ug                                str
codigo_modalidade_compra               str
modalidade_compra                      str
numero_processo                        str
objeto                                 str
situacao_licitacao                     str
codigo_orgao_superior                  str
nome_orgao_superior                    str
codigo_orgao                           str
nome_orgao                             str
uf                                     str
municipio                              str
data_resultado_compra       datetime64[us]
data_abertura               datetime64[us]
valor_licitacao                    float64
ano_mes_arquivo                        str
dtype: object

Valor Licitação: min=0.00, max=2,695,903,800.00, mediana=5,000.00

Flag Vencedor após tratamento:
flag_vencedor
False    3489950
True      949377
Name: count, dtype: int64

Duplicatas na chav